In [51]:
import cv2
import numpy as np

video_path = '/content/denis.mp4'
background_img_path = '/content/imgg.png'
output_path = 'output_assignment.mp4'

In [52]:
cap = cv2.VideoCapture(video_path)
bg_image = cv2.imread(background_img_path)

if not cap.isOpened() or bg_image is None:
    raise ValueError("Error: Could not open video or image.")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

bg_image = cv2.resize(bg_image, (frame_width, frame_height))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

ret, frame = cap.read()
average_frame = np.float32(frame)
frame_count = 0

print("Initialization complete.")

Initialization complete.


In [53]:
while True:
    ret, current_frame = cap.read()
    if not ret:
        break

    frame_count += 1
    learning_rate = 0.1 if frame_count < 30 else 0.004

    cv2.accumulateWeighted(current_frame, average_frame, learning_rate)
    background_model = cv2.convertScaleAbs(average_frame)
    difference = cv2.absdiff(current_frame, background_model)
    gray_diff = cv2.cvtColor(difference, cv2.COLOR_BGR2GRAY)

    gray_diff = cv2.GaussianBlur(gray_diff, (5, 5), 0)
    _, mask = cv2.threshold(gray_diff, 25, 255, cv2.THRESH_BINARY)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.dilate(mask, kernel_dilate, iterations=1)

    mask = cv2.GaussianBlur(mask, (5, 5), 0)

    mask_3ch = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR) / 255.0

    foreground = current_frame * mask_3ch
    background_part = bg_image * (1 - mask_3ch)

    result = cv2.add(foreground.astype(np.uint8), background_part.astype(np.uint8))
    out.write(result)

In [54]:
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Final output saved to: {output_path}")

Final output saved to: output_assignment.mp4
